In [1]:
import os

In [2]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [3]:

# ".." tells Python to step out of the current folder into the parent folder
os.chdir("d:\\PredictBot-Score-MLOps")

# Check where you are now


In [4]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [5]:
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import pandas as pd 
import io
import os
from src.predictor_bot_score.utils.src_util_s3_  import (save_manifest , 
                                                         save_local ,
                                                         self_local_save_mainfest , 
                                                         save_file_s3 ,
                                                         self_s3_mainfest,
                                                         s3_login)
from botocore.exceptions import  ClientError



In [6]:
@dataclass(frozen=True)
class DataTransformationConfig:
    validated_data_path: Path
    transformed_data_dir: Path
    bucket_name : str

In [7]:
class Config_manager:

    def __init__(self , config = CONFIG_PATH):

        self.config_path = yaml_load(config)

        create_directories([self.config_path.artifacts_root])
    
    def get_data_transformation_config(self):

        config = self.config_path.data_transformation

        create_directories([config.transformed_data])

        data_transformation_config = DataTransformationConfig(
            validated_data_path=Path(config.validated_data_path),
            transformed_data_dir =Path(config.transformed_data),
            bucket_name = self.config_path.s3_config.bucket_name
        )
        
        return data_transformation_config




In [ ]:
class Data_transformation:

    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.BUCKET_NAME = self.config.bucket_name
        self.s3 = s3_login()
        self.data = self.read_data()
        

    def read_data(self):
        try:

            keys = []

            logger.info("=" * 50)
            logger.info("DATA TRANSFORMATION PIPELINE STARTED")
            logger.info("=" * 50)
            
            for page in self.s3.list_objects_v2(Bucket=self.BUCKET_NAME,Prefix='combined_data')['Contents']:
                if page.get('Key').endswith('.csv'):
                    keys.append(page.get('Key'))

            logger.info("S3 . Connection exists ")

            combined_file_key = sorted(keys)[-1]

            return combined_file_key
        
        except ClientError as e:

            # This is your custom message
            print("--- ALERT: The file is missing! Please check the path. ---")
            
            # This is the log file entry
            logger.error(f"File not found at: {self.BUCKET_NAME}")
            raise

            
             

    def transformed_data(self ):
        
        try:

            file_key = self.data
            obj = self.s3.get_object(Bucket=self.BUCKET_NAME,Key=file_key)
            df = pd.read_csv(io.BytesIO(obj['Body'].read()))

            df["timestamp"] = pd.to_datetime(self.data["timestamp"], utc=True)

            df["bot_score"] = df["bot_score"].astype(float)

            timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            out_path  = self.config.transformed_data_dir / f"_{timestamp}.csv"

            
            
            logger.info(f"Data saved  {out_path}")
            logger.info(f"DATA TRANSFORMATION DONE")
            
            return df
        
        except Exception as e:
            logger.info(e)

In [9]:
transformation_config = Config_manager().get_data_transformation_config()
transformation_config = Data_transformation(transformation_config)
transformation_config.transformed_data('combined_data/run__2026_07_10_16_12/combined_df.csv')

[2026-07-10 19:46:07,717: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-10 19:46:07,719: INFO: common: Directory created (or already exists) at: artifacts]
[2026-07-10 19:46:07,721: INFO: common: Directory created (or already exists) at: artifacts/data_transformation/transformed_data]
[2026-07-10 19:46:08,235: INFO: 179799501: ==================================================]
[2026-07-10 19:46:08,236: INFO: 179799501: DATA TRANSFORMATION PIPELINE STARTED]
[2026-07-10 19:46:08,237: INFO: 179799501: ==================================================]
[2026-07-10 19:46:10,175: INFO: 179799501: S3 . Connection exists ]
[2026-07-10 19:46:14,989: INFO: 179799501: string indices must be integers, not 'str']


In [13]:
content = yaml_load(Path('config\config.yaml'))

[2026-07-10 18:54:13,417: INFO: common: yaml file: config\config.yaml loaded successfully]


In [20]:
content.s3_config.bucket_name

'predict-bot-mlops'